<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/03-RAG_with_LlamaIndex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Packages and Setup Variables


In [ ]:
!pip install -q openai==1.107.0 google-genai==1.36.0 llama-index==0.14.0 llama-index-llms-google-genai==0.3.0 jedi==0.19.2

In [ ]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "<YOUR_API_KEY>"
# os.environ["GOOGLE_API_KEY"] = "<YOUR_API_KEY>"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('Google_api_key')
except ImportError:
    pass  # Running locally — keys are expected in the environment

# Load Dataset


## Download


The dataset includes several articles from the TowardsAI blog, which provide an in-depth explanation of the LLaMA2 model.


In [ ]:
!curl -o ./mini-dataset.csv https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv

## Read File


In [ ]:
import csv

rows = []

# Load the CSV file
with open("./mini-dataset.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            continue
            # Skip header row
        rows.append(row)

# The number of characters in the dataset.
print("number of articles:", len(rows))

# Generate Embedding


In [ ]:
from llama_index.core import Document

# Convert the texts to Document objects so the LlamaIndex framework can process them.
documents = [Document(text=row[1]) for row in rows]

In [ ]:
documents[0]

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding


# Build index / generate embeddings using OpenAI embedding model
index = VectorStoreIndex.from_documents(
    documents,
    transformations=[SentenceSplitter(chunk_size=768, chunk_overlap=64)],
    embed_model=OpenAIEmbedding(model="text-embedding-3-small"),
    show_progress=True,
)

In [ ]:
# Visualize Chunks and Chunks Overlap after the Sentence Splitter Transformation

documents = index.docstore.docs
for doc in documents.values():
  print(doc.text)
  print("-_-_-_-_-_-_-_-_")

# Query Dataset


In [ ]:
# # Define a query engine that is responsible for retrieving related pieces of text,
# # and using a LLM to formulate the final answer.

from llama_index.llms.google_genai import GoogleGenAI
import google.genai.types as types

config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_budget=0),
    max_output_tokens=512,
    temperature=1,
)

llm = GoogleGenAI(
    model="gemini-2.5-flash",
    generation_config=config,
    )

query_engine = index.as_query_engine(llm=llm)

In [ ]:
response = query_engine.query("How many parameters LLaMA 2 model has?")
print(response)

In [ ]:
response = query_engine.query("When will Llama 4 be released?")
print(response)

## Optional: Retrieved Source Nodes Inspector

Inspect which source chunks were retrieved for each query, along with their similarity scores.
This helps you understand what context the model is using to generate its answers.

In [ ]:
# Use a retriever directly to inspect retrieved nodes
retriever = index.as_retriever(similarity_top_k=3)

inspect_query = "How many parameters LLaMA 2 model has?"
nodes = retriever.retrieve(inspect_query)

print(f"Query: {inspect_query}\n")
for i, node in enumerate(nodes):
    print(f"--- Source {i+1} (score: {node.score:.4f}) ---")
    print(node.text[:300])
    print()

## Optional: Top-K Comparison

Compare how retrieval quality changes with different `similarity_top_k` values.
Increasing top_k retrieves more context but may introduce noise.

In [ ]:
test_query = "What are the key features of LLaMA 2?"

for top_k in [1, 3, 5]:
    retriever_k = index.as_retriever(similarity_top_k=top_k)
    nodes_k = retriever_k.retrieve(test_query)
    qe_k = index.as_query_engine(
        llm=llm,
        similarity_top_k=top_k,
    )
    resp_k = qe_k.query(test_query)
    print(f"=== top_k={top_k} | chunks retrieved: {len(nodes_k)} ===")
    print(str(resp_k)[:400])
    print()